# Brisbane Plot

A Brisbane plot visualizes the local *density* of independent, genome-wide-significant GWAS signals along the genome, rather than their -log10(p-value). Each point represents a genomic window and its height/color reflects how many independent signals fall within it, making it easy to spot regions of unusually dense (or sparse) association signal.

In this notebook we show how to use `IDEAL-GENOM` to generate a Brisbane plot with `brisbane_draw`.

In [ ]:
import sys
import os

import pandas as pd

In [ ]:
# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

In [ ]:
from ideal_genom.visualizations.manhattan_type import brisbane_draw, brisbane_process_data
from ideal_genom.core.get_examples import get_yengo_height_independent_signals

The data is taken from the article:

Yengo, L., Vedantam, S., Marouli, E., Sidorenko, J., Bartell, E., Sakaue, S., ... & Lee, J. Y. (2022). A saturated map of common genetic variants associated with human height. *Nature*, **610**, 704-712. https://doi.org/10.1038/s41586-022-05275-y

We use Supplementary Table 5 of the paper, which lists the 12,111 conditionally independent (COJO) genome-wide-significant SNPs identified in the GIANT/UK Biobank height meta-analysis, on genome build hg19/GRCh37. This is the same table used to generate the original Brisbane plot in the paper (Fig. 2).

If the table has not been downloaded already, we proceed to fetch it or we get the path to it.

In [ ]:
signals_path = get_yengo_height_independent_signals()

In [ ]:
df_signals = pd.read_csv(signals_path, sep='\t')

Let us look at the columns available in the independent signals table.

In [ ]:
print("Number of independent signals: ", df_signals.shape[0])
print("Columns: ", df_signals.columns.to_list())
df_signals.head()

Here we draw a Brisbane plot using 100 kb windows, the same window size the original paper used to define its own precomputed `SIGNAL_DENSITY` column.

In [ ]:
brisbane_draw(
    data_df=df_signals,
    chr_col='CHR',
    pos_col='POS',
    plot_dir=signals_path.parent.as_posix(),
    window_kb=100,
    save_name='brisbane_plot_100kb.png',
    point_size=10,
    figsize=(11, 3),
)

Widening the window smooths out the density signal and highlights broader regions of the genome with many independent hits (e.g. the height-associated region near the *HMGA2*/*ZBTB20* loci).

In [ ]:
brisbane_draw(
    data_df=df_signals,
    chr_col='CHR',
    pos_col='POS',
    plot_dir=signals_path.parent.as_posix(),
    window_kb=1000,
    ytick_step=None,
    save_name='brisbane_plot_1000kb.png',
    chr_colors=['#377eb8', '#4daf4a'],
    figsize=(11, 3)
)

Finally, `brisbane_process_data` can be used directly to inspect the underlying windowed counts. Note that the paper's `SIGNAL_DENSITY` counts other COJO SNPs within a *sliding* 100 kb window centered on each individual SNP, whereas `brisbane_process_data` bins signals into *fixed, non-overlapping* 100 kb windows — so the two are not expected to match exactly, but should be of a similar order of magnitude.

In [ ]:
brisbane_data = brisbane_process_data(df_signals, chr_col='CHR', pos_col='POS', window_kb=100)

print("Max signals in a single 100kb window (recomputed):", brisbane_data['max_count'])
print("Max signals in a single 100kb window (paper's SIGNAL_DENSITY + 1):", df_signals['SIGNAL_DENSITY'].max() + 1)